# MLOps Training 2026/2027 — Task 2

## Notebook 1 — Read & Join the Tables

This notebook reads the Olist PostgreSQL tables, checks their grain and key quality, aggregates every one-to-many table before joining, and saves one row per `order_id`. It does not create the late-delivery label yet; that belongs to Notebook 2.

**Input:** PostgreSQL database created in Task 1.

**Output artifact:** `Task2/artifacts/01_joined_ml_table.parquet`

**Important:** Reviews and actual delivery dates are retained only as analysis/label-support columns. They must not be used as prediction features before delivery.


## 1. Environment and paths

The notebook uses environment variables for the database password. If `OLIST_DB_PASSWORD` is not set, it asks for the password interactively.


In [1]:
from pathlib import Path
import os
import getpass
import json
import pandas as pd
from sqlalchemy import create_engine, text, inspect

PROJECT_ROOT = Path(r"C:\Users\HP\Desktop\MLOps")
TASK1_ROOT = PROJECT_ROOT / "Task1"
TASK2_ROOT = PROJECT_ROOT / "Task2"
ARTIFACTS_DIR = TASK2_ROOT / "artifacts"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

DB_HOST = os.getenv("OLIST_DB_HOST", "localhost")
DB_PORT = os.getenv("OLIST_DB_PORT", "5432")
DB_NAME = os.getenv("OLIST_DB_NAME", "olist")
DB_USER = os.getenv("OLIST_DB_USER", "postgres")
DB_PASSWORD = os.getenv("OLIST_DB_PASSWORD") or getpass.getpass("PostgreSQL password: ")
engine = create_engine(f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}", pool_pre_ping=True)

with engine.connect() as connection:
    print(connection.execute(text("SELECT version();")).scalar_one())
print(f"Artifacts directory: {ARTIFACTS_DIR}")


PostgreSQL 17.10 on x86_64-windows, compiled by msvc-19.44.35226, 64-bit
Artifacts directory: C:\Users\HP\Desktop\MLOps\Task2\artifacts


## 2. Read each table and document its grain

Every source table is read independently first. This makes the row meaning, size, columns, missingness, and candidate keys visible before any joins are attempted.


In [2]:
inspector = inspect(engine)
expected_tables = [
    "olist_customers", "olist_geolocation", "olist_orders",
    "olist_order_items", "olist_order_payments", "olist_order_reviews",
    "olist_products", "olist_sellers", "product_category_name_translation"
]
actual_tables = set(inspector.get_table_names())
missing_tables = set(expected_tables) - actual_tables
if missing_tables:
    raise AssertionError(f"Missing Task 1 tables: {sorted(missing_tables)}")

tables = {}
table_summary = []
for table_name in expected_tables:
    tables[table_name] = pd.read_sql_query(f'SELECT * FROM "{table_name}"', engine)
    table_summary.append({
        "table_name": table_name,
        "rows": len(tables[table_name]),
        "columns": len(tables[table_name].columns),
        "duplicate_rows": int(tables[table_name].duplicated().sum()),
    })

table_summary = pd.DataFrame(table_summary).sort_values("table_name").reset_index(drop=True)
display(table_summary)


,table_name,rows,columns,duplicate_rows
0,olist_customers,99441,5,0
1,olist_geolocation,1000163,5,261831
2,olist_order_items,112650,7,0
3,olist_order_payments,103886,5,0
4,olist_order_reviews,99224,7,0
5,olist_orders,99441,8,0
6,olist_products,32951,9,0
7,olist_sellers,3095,4,0
8,product_category_name_translation,71,2,0


In [3]:
def key_quality(df, key_columns, table_name):
    result = {
        "table_name": table_name,
        "key": ", ".join(key_columns),
        "rows": len(df),
        "unique_keys": int(df[key_columns].drop_duplicates().shape[0]),
        "null_key_rows": int(df[key_columns].isna().any(axis=1).sum()),
    }
    result["duplicate_key_rows"] = result["rows"] - result["unique_keys"]
    return result

key_checks = pd.DataFrame([
    key_quality(tables["olist_customers"], ["customer_id"], "olist_customers"),
    key_quality(tables["olist_orders"], ["order_id"], "olist_orders"),
    key_quality(tables["olist_products"], ["product_id"], "olist_products"),
    key_quality(tables["olist_sellers"], ["seller_id"], "olist_sellers"),
    key_quality(tables["olist_order_items"], ["order_id", "order_item_id"], "olist_order_items"),
    key_quality(tables["olist_order_payments"], ["order_id", "payment_sequential"], "olist_order_payments"),
])
display(key_checks)


,table_name,key,rows,unique_keys,null_key_rows,duplicate_key_rows
0,olist_customers,customer_id,99441,99441,0,0
1,olist_orders,order_id,99441,99441,0,0
2,olist_products,product_id,32951,32951,0,0
3,olist_sellers,seller_id,3095,3095,0,0
4,olist_order_items,"order_id, order_item_id",112650,112650,0,0
5,olist_order_payments,"order_id, payment_sequential",103886,103886,0,0


## 3. Aggregate one-to-many tables before joining

The order table is the base grain. Items, payments, and reviews can contain multiple rows per order, so each is summarized by `order_id` independently. This prevents fan-out and duplicated sums.


In [4]:
orders = tables["olist_orders"].copy()
customers = tables["olist_customers"].copy()
items = tables["olist_order_items"].copy()
payments = tables["olist_order_payments"].copy()
reviews = tables["olist_order_reviews"].copy()
products = tables["olist_products"].copy()
sellers = tables["olist_sellers"].copy()

items_products = items.merge(products, on="product_id", how="left", validate="many_to_one")
items_sellers = items_products.merge(sellers, on="seller_id", how="left", validate="many_to_one")

items_agg = (items_sellers.groupby("order_id", as_index=False)
    .agg(
        num_items=("order_item_id", "count"),
        num_distinct_products=("product_id", "nunique"),
        total_price=("price", "sum"),
        total_freight=("freight_value", "sum"),
        num_sellers=("seller_id", "nunique"),
        num_product_categories=("product_category_name", "nunique"),
        avg_product_weight_g=("product_weight_g", "mean"),
        avg_product_photos_qty=("product_photos_qty", "mean"),
        seller_state_count=("seller_state", "nunique"),
    ))

payments_agg = (payments.groupby("order_id", as_index=False)
    .agg(
        payment_record_count=("payment_sequential", "count"),
        total_payment_value=("payment_value", "sum"),
        max_payment_installments=("payment_installments", "max"),
    ))

reviews_agg = (reviews.groupby("order_id", as_index=False)
    .agg(
        review_record_count=("review_id", "nunique"),
        avg_review_score=("review_score", "mean"),
    ))

print("Aggregated shapes:")
print({"items": items_agg.shape, "payments": payments_agg.shape, "reviews": reviews_agg.shape})


Aggregated shapes:
{'items': (98666, 10), 'payments': (99440, 4), 'reviews': (98673, 3)}


## 4. Join to one row per order

Only one-to-one or already aggregated tables are joined to `orders`. The `validate` arguments make unexpected relationship violations fail loudly.


In [5]:
joined = (orders
    .merge(customers[["customer_id", "customer_unique_id", "customer_zip_code_prefix", "customer_city", "customer_state"]], on="customer_id", how="left", validate="one_to_one")
    .merge(items_agg, on="order_id", how="left", validate="one_to_one")
    .merge(payments_agg, on="order_id", how="left", validate="one_to_one")
    .merge(reviews_agg, on="order_id", how="left", validate="one_to_one"))

assert joined["order_id"].is_unique, "The joined table is not one row per order_id."
assert len(joined) == len(orders), "A left join changed the number of order rows."

print(f"Joined shape: {joined.shape}")
print(f"Unique order IDs: {joined.order_id.nunique():,}")
display(joined.head())


Joined shape: (99441, 26)
Unique order IDs: 99,441


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,...,num_sellers,num_product_categories,avg_product_weight_g,avg_product_photos_qty,seller_state_count,payment_record_count,total_payment_value,max_payment_installments,review_record_count,avg_review_score
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,7c396fd4830fd04220f754e42b4e5bff,3149,...,1.0,1.0,500.0,4.0,1.0,3.0,38.71,1.0,1.0,4.0
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00,af07308b275d755c9edb36a90c618231,47813,...,1.0,1.0,400.0,1.0,1.0,1.0,141.46,1.0,1.0,4.0
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00,3a653a41f6f9fc3d2a113cf8398680e8,75265,...,1.0,1.0,420.0,1.0,1.0,1.0,179.12,3.0,1.0,5.0
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00,7c142cf63193a1473d2e66489a9ae977,59296,...,1.0,1.0,450.0,3.0,1.0,1.0,72.20,1.0,1.0,5.0
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00,72632f0f9dd73dfee390c9b22eb56dd6,9195,...,1.0,1.0,250.0,4.0,1.0,1.0,28.62,1.0,1.0,5.0


## 5. Quality checks and artifact save

The artifact is saved in Parquet format so later notebooks can load it without reconnecting to PostgreSQL. A small metadata JSON records the row count, columns, and aggregation decisions.


In [6]:
artifact_path = ARTIFACTS_DIR / "tables" / "01_joined_ml_table.parquet"
metadata_path = ARTIFACTS_DIR / "reports" / "01_joined_ml_table_metadata.json"

joined.to_parquet(artifact_path, index=False)
metadata = {
    "artifact": artifact_path.name,
    "grain": "one row per order_id",
    "rows": int(len(joined)),
    "columns": list(joined.columns),
    "source_tables": expected_tables,
    "aggregations": {
        "order_items": "aggregated by order_id before joining",
        "order_payments": "aggregated by order_id before joining",
        "order_reviews": "aggregated by order_id before joining",
        "products_and_sellers": "joined to items with many_to_one validation, then aggregated by order_id",
    },
    "analysis_only_columns": ["order_delivered_customer_date", "avg_review_score", "review_record_count"],
}
metadata_path.write_text(json.dumps(metadata, indent=2), encoding="utf-8")
print(f"Saved: {artifact_path}")
print(f"Saved: {metadata_path}")


Saved: C:\Users\HP\Desktop\MLOps\Task2\artifacts\tables\01_joined_ml_table.parquet
Saved: C:\Users\HP\Desktop\MLOps\Task2\artifacts\reports\01_joined_ml_table_metadata.json


## 6. Final completion check

Notebook 1 is complete only if the artifact exists, has one row per order, has the same number of rows as the orders table, and contains the required order/customer/aggregation fields.


In [7]:
required_columns = {
    "order_id", "customer_id", "order_status", "customer_state",
    "num_items", "total_price", "total_freight", "total_payment_value"
}
assert artifact_path.exists() and artifact_path.stat().st_size > 0
assert joined["order_id"].is_unique
assert required_columns.issubset(joined.columns)
reloaded = pd.read_parquet(artifact_path)
assert reloaded.shape == joined.shape
assert reloaded["order_id"].is_unique
print("ALL NOTEBOOK 1 CHECKS PASSED")
print(f"Artifact rows: {len(reloaded):,}")
print(f"Artifact columns: {len(reloaded.columns)}")


ALL NOTEBOOK 1 CHECKS PASSED
Artifact rows: 99,441
Artifact columns: 26
